# 06 — Robustness

Each variation re-estimates demand (full sample and pre-merger sample) and re-simulates both mergers. Reported:
price sensitivity, nesting parameter, passenger-weighted own-price elasticity, median Lerner index, the predicted
Alaska–Hawaiian fare change (pre-merger estimates, 2023Q4–2024Q2 base) next to the realised DiD change, and the
predicted JetBlue–Spirit fare change in 2025Q2. The linear nested logit with one air-travel nest is the workhorse
here because it can be re-estimated in seconds; random-coefficient robustness (starting values, simulation draws)
is in section 6.

1. Instruments: every family from notebook 03, plus **exogenous changes in market structure** (course notes,
   Week 8): the court-ordered unwinding of the American–JetBlue Northeast Alliance (decision May 2023; wound down
   from 2023Q3) and the Alaska–Hawaiian closing, both of which shift markups of the affected carriers' products.
2. Fixed effects: carrier × quarter; distance band × quarter.
3. Market definition: directional airport pairs (by trip origin); city-market pairs.
4. Nesting: nonstop/connecting nests; air-travel nest with $\rho$ fixed at 0.2, 0.4, 0.6, 0.8.
5. Market size: potential travellers equal to 0.5 and 2 times the geometric-mean population.
6. Random coefficients: five starting values and 200 instead of 50 simulation draws.

In [ ]:
%run 00_setup.ipynb

In [ ]:
%run 00_model_functions.ipynb

In [ ]:
ASHA, B6NK = ("AS", "HA"), ("B6", "NK")
BASE_ASHA = {qlabel(*q) for q in [(2023, 4), (2024, 1), (2024, 2)]}
LATEST = qlabel(*LAST_Q)
val = load_json("asha_validation")
REALISED = [d for d in val["did"] if "same seasons" in d["specification"]][0]["% change"]
print("realised AS/HA change (DiD, %):", round(REALISED, 2))
NEA_AIRPORTS = {"JFK", "LGA", "EWR", "BOS"}

def add_structure_ivs(p):
    p = p.copy()
    nea_route = (p.a.isin(NEA_AIRPORTS) | p.b.isin(NEA_AIRPORTS))
    both = p.groupby("market_ids").carrier.transform(lambda s: {"AA", "B6"} <= set(s))
    p["iv_nea_unwind"] = (nea_route & both & p.carrier.isin(["AA", "B6"]) & (p.t >= qindex(2023, 3))).astype(float)
    ov = p.groupby("market_ids").carrier.transform(lambda s: {"AS", "HA"} <= set(s))
    p["iv_asha_close"] = (ov & p.carrier.isin(["AS", "HA"]) & (p.t >= qindex(2024, 4))).astype(float)
    return p

def load(definition):
    if definition == "airport_nondir":
        p = pd.read_parquet(INTERIM / "products_airport_nondir_iv.parquet")
    else:
        p = add_instruments(pd.read_parquet(CLEAN / f"products_{definition}.parquet"))
    if SMOKE:
        keep = set(p.loc[p.carrier.isin(["AS", "HA", "B6", "NK"]), "route"].unique()[:150]) | set(p.route.unique()[:200])
        p = p[p.route.isin(keep)]
    return add_structure_ivs(p).sort_values(["market_ids", "carrier", "nonstop"]).reset_index(drop=True)

def overlap(d, parties, quarters):
    both = d[d.quarter.isin(quarters)].groupby("market_ids").carrier.agg(lambda s: set(parties) <= set(s))
    return both.index[both].tolist()

def predicted_change(kind, prm, d, parties, quarters, iv):
    mk = overlap(d, parties, quarters)
    if not mk:
        return np.nan, 0
    sub = d[d.market_ids.isin(mk)].reset_index(drop=True)
    r = evaluate_fixed(sub, kind, prm, iv)
    sim, cs = simulate_merger(r, sub, parties, markets=mk)
    return party_avg(route_summary(sim, cs, parties)), len(mk)

def run_spec(label, d, iv, nest="air", fe=("route",), dummies=("quarter", "carrier_fe"), rho_fixed=None, post=True):
    t0 = time.time()
    kind = {"air": "nested_air", "service": "nested_service", None: "logit"}[nest]
    full = linear_iv(d, iv, nest=nest, fe=fe, dummies=dummies, rho_fixed=rho_fixed)
    pre = d[d.t <= qindex(2024, 2)].reset_index(drop=True)
    prer = linear_iv(pre, iv, nest=nest, fe=fe, dummies=dummies, rho_fixed=rho_fixed)
    row = {"specification": label, "model": kind, "alpha": -full["alpha"], "alpha s.e.": full["alpha_se"],
           "rho": full["rho"] if nest else np.nan, "J": full["J"], "own elasticity": np.nan, "median Lerner": np.nan,
           "AS/HA predicted (%)": np.nan, "B6/NK predicted (%)": np.nan}
    valid = full["alpha"] < 0 and prer["alpha"] < 0 and (nest is None or (0 <= full["rho"] < 1 and 0 <= prer["rho"] < 1))
    if valid:
        prm = {"alpha": full["alpha"], "rho": full["rho"]}
        if post:
            r = evaluate_fixed(d, kind, prm, iv)
            pe = post_estimation(r, d)
            row["own elasticity"] = float(np.average(pe.own_elasticity, weights=pe.passengers))
            row["median Lerner"] = float(pe.lerner.median())
        row["AS/HA predicted (%)"], _ = predicted_change(kind, {"alpha": prer["alpha"], "rho": prer["rho"]}, pre, ASHA, BASE_ASHA, iv)
        row["B6/NK predicted (%)"], _ = predicted_change(kind, prm, d, B6NK, {LATEST}, iv)
    row["minutes"] = (time.time() - t0) / 60
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in row.items()})
    return row

base = load("airport_nondir")
IVB = IV_SETS["baseline"]
rows = []

## 1. Instruments

In [ ]:
for name, ivs in IV_SETS.items():
    rows.append(run_spec(f"IV: {name}", base, ivs))
rows.append(run_spec("IV: baseline + market-structure changes", base, IVB + ["iv_nea_unwind", "iv_asha_close"]))
rows.append(run_spec("IV: baseline, logit", base, IVB, nest=None))

## 2. Fixed effects

In [ ]:
base["carrier_q"] = base.carrier_fe + "_" + base.quarter
base["band_q"] = base.dist_band.astype(str) + "_" + base.quarter
rows.append(run_spec("FE: route + carrier x quarter", base, IVB, fe=("route", "carrier_q"), dummies=(), post=False))
rows.append(run_spec("FE: route + distance band x quarter + carrier", base, IVB, fe=("route", "band_q"), dummies=("carrier_fe",), post=False))

## 3. Market definition

In [ ]:
for definition, label in [("airport_dir", "Market: directional airport pairs"), ("city_nondir", "Market: city-market pairs")]:
    d = load(definition)
    print(label, f"{len(d):,} products, {d.market_ids.nunique():,} markets")
    rows.append(run_spec(label, d, IVB))
    del d

## 4. Nesting structure and parameter

In [ ]:
rows.append(run_spec("Nest: nonstop vs connecting", base, IVB, nest="service"))
for r_ in [0.2, 0.4, 0.6, 0.8]:
    rows.append(run_spec(f"Nest: air travel, rho fixed at {r_}", base, IVB, rho_fixed=r_))

## 5. Market size

In [ ]:
for scale in [0.5, 2.0]:
    d = base.assign(market_size=base.market_size * scale, shares=base.shares / scale)
    rows.append(run_spec(f"Market size x {scale}", d, IVB))

## Summary table

In [ ]:
allrows = pd.DataFrame(rows)
allrows.to_csv(RES / "robustness_all.csv", index=False)
show = allrows.set_index("specification")[["alpha", "alpha s.e.", "rho", "own elasticity", "median Lerner",
                                           "AS/HA predicted (%)", "B6/NK predicted (%)"]]
save_table(show, "robustness", float_format="%.3f",
           note="Nested logit with one air-travel nest unless stated; each row re-estimates demand on the full and pre-merger samples "
                f"and re-simulates both mergers. Realised AS/HA change (DiD): {REALISED:.2f}\\%. Blank cells: the specification implies "
                "upward-sloping demand or $\\rho \\notin [0,1)$, so no simulation is reported.")
show

In [ ]:
plot = allrows.dropna(subset=["AS/HA predicted (%)"])
fig, ax = plt.subplots(figsize=(7.5, 0.28 * len(plot) + 1.2))
y = np.arange(len(plot))
ax.scatter(plot["AS/HA predicted (%)"], y, label="AS/HA predicted", color=PALETTE[0])
ax.scatter(plot["B6/NK predicted (%)"], y, label="B6/NK predicted", color=PALETTE[1], marker="s")
ax.axvline(REALISED, color=PALETTE[0], ls="--", lw=1, label="AS/HA realised (DiD)")
ax.set_yticks(y); ax.set_yticklabels(plot.specification, fontsize=7); ax.invert_yaxis()
ax.set_xlabel("Passenger-weighted fare change of merging carriers (%)"); ax.legend(fontsize=8)
fig.tight_layout(); savefig(fig, "robustness_predictions")

## 6. Random coefficients: starting values and simulation draws

In [ ]:
est = load_json("demand_full_sample")
starts = pd.read_csv(RES / "rc_starting_values.csv")
display(starts)
rc_routes = est["rc_routes"][: (60 if SMOKE else 300)]
p_rc = base[base.route.isin(rc_routes)].reset_index(drop=True)
draw_rows = []
for draws in [50, 200]:
    prob = make_problem(p_rc, "rc", IVB, draws=draws, seed=7)
    with pyblp.parallel(N_CORES):
        r = solve_rc(prob, "rc", sigma0=est["rc_demand"]["sigma"], pi0=est["rc_demand"]["pi"], method="1s",
                     optimization=pyblp.Optimization("l-bfgs-b", {"gtol": 1e-5, "maxiter": 60}))
    draw_rows.append({"draws": draws, **{k: v for k, v in rc_params(r).items() if k != "beta"}, "seconds": r.elapsed})
    print(draws, np.round(np.diag(r.sigma), 4), np.round(np.ravel(r.pi), 4))
draws_tab = pd.DataFrame(draw_rows)
save_table(draws_tab.set_index("draws")[["objective", "sigma", "pi", "converged"]].astype(str), "robust_rc_draws",
           note=f"Random-coefficients demand re-estimated on {len(rc_routes)} routes with 50 and 200 simulation draws per market (new seed).")
starts_tab = starts[["start", "sigma0", "objective", "sigma", "pi", "converged"]].astype(str).set_index("start")
save_table(starts_tab, "robust_starting_values", note="Random-coefficients demand (notebook 03 sample) from three starting values; one-step GMM.")
draws_tab